In [5]:
import sys
sys.path.append('.')  # models/ and experiments/ are inside notebooks/

from experiments.train_cnn_loso import (
    WESADDataset,
    SUBJECTS,
    load_subject,
    load_all_except,
    get_val_subject,
    train_one_fold
)

from models.cnn_lstm_encoder import MultiModalCNNLSTM

import torch
import numpy as np
import pandas as pd
import time
import os
import pickle

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score
)

from torch.utils.data import DataLoader

In [6]:
import sys
sys.path.append('.')  # models/ and experiments/ are inside notebooks/

from experiments.train_cnn_loso import (
    WESADDataset,
    SUBJECTS,
    load_subject,
    load_all_except,
    get_val_subject,
    train_one_fold
)

from models.cnn_lstm_encoder import MultiModalCNNLSTM

import torch
import numpy as np
import pandas as pd
import time
import os
import pickle

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score
)

from torch.utils.data import DataLoader

In [7]:
torch.manual_seed(42)

test_sid = 2

val_sid = get_val_subject(test_sid)

train_X, train_y = load_all_except(
    test_sid,
    val_sid
)

val_X, val_y = load_subject(val_sid)

train_loader = DataLoader(
    WESADDataset(train_X, train_y),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    WESADDataset(val_X, val_y),
    batch_size=32,
    shuffle=False
)

model = MultiModalCNNLSTM().to(device)

start = time.time()

model, best_f1 = train_one_fold(
    model,
    train_loader,
    val_loader,
    device,
    epochs=5,
    patience=8
)

elapsed = time.time() - start

print(
    f"5 epochs took {elapsed:.1f}s "
    f"-> estimated 30 epochs x 15 folds: "
    f"{(elapsed/5)*30*15/60:.1f} min"
)

5 epochs took 121.8s -> estimated 30 epochs x 15 folds: 182.7 min


In [8]:
all_results_lstm = []

all_predictions_lstm = {}

EPOCHS = 30

start_total = time.time()

In [9]:
for fold_idx, test_sid in enumerate(SUBJECTS):

    print(
        f"\n=== CNN-LSTM Fold "
        f"{fold_idx+1}/{len(SUBJECTS)} "
        f"| Test subject S{test_sid} ==="
    )

    val_sid = get_val_subject(test_sid)

    torch.manual_seed(42)
    np.random.seed(42)

    train_X, train_y = load_all_except(
        test_sid,
        val_sid
    )

    val_X, val_y = load_subject(val_sid)

    test_X, test_y = load_subject(test_sid)

    train_loader = DataLoader(
        WESADDataset(train_X, train_y),
        batch_size=32,
        shuffle=True
    )

    val_loader = DataLoader(
        WESADDataset(val_X, val_y),
        batch_size=32,
        shuffle=False
    )

    test_loader = DataLoader(
        WESADDataset(test_X, test_y),
        batch_size=32,
        shuffle=False
    )

    model = MultiModalCNNLSTM().to(device)

    fold_start = time.time()

    model, best_val_f1 = train_one_fold(
        model,
        train_loader,
        val_loader,
        device,
        epochs=EPOCHS,
        patience=8
    )

    fold_time = time.time() - fold_start

    model.eval()

    test_preds = []
    test_probs = []
    test_true = []

    with torch.no_grad():

        for Xb, yb in test_loader:

            out = model(Xb.to(device))

            probs = torch.softmax(
                out,
                dim=1
            )[:, 1]

            preds = out.argmax(dim=1)

            test_preds.extend(
                preds.cpu().numpy()
            )

            test_probs.extend(
                probs.cpu().numpy()
            )

            test_true.extend(
                yb.numpy()
            )

    acc = accuracy_score(
        test_true,
        test_preds
    )

    f1 = f1_score(
        test_true,
        test_preds,
        average="macro",
        zero_division=0
    )

    try:
        auroc = roc_auc_score(
            test_true,
            test_probs
        )
    except ValueError:
        auroc = np.nan

    all_results_lstm.append({
        "subject": test_sid,
        "val_subject": val_sid,
        "val_f1": best_val_f1,
        "accuracy": acc,
        "f1_macro": f1,
        "auroc": auroc,
        "train_time_sec": fold_time
    })

    all_predictions_lstm[test_sid] = {
        "y_true": np.array(test_true),
        "y_pred": np.array(test_preds),
        "y_prob": np.array(test_probs)
    }

    torch.save(
        model.state_dict(),
        f"../results/models/cnnlstm_S{test_sid}.pt"
    )

    print(
        f"  -> Acc={acc:.4f} "
        f"F1={f1:.4f} "
        f"AUROC={auroc:.4f} "
        f"| fold_time={fold_time:.1f}s"
    )


=== CNN-LSTM Fold 1/15 | Test subject S2 ===
  -> Acc=0.7571 F1=0.7278 AUROC=0.8027 | fold_time=527.1s

=== CNN-LSTM Fold 2/15 | Test subject S3 ===
  -> Acc=0.8732 F1=0.8260 AUROC=0.9324 | fold_time=539.3s

=== CNN-LSTM Fold 3/15 | Test subject S4 ===
  -> Acc=0.9301 F1=0.9072 AUROC=0.9981 | fold_time=184.0s

=== CNN-LSTM Fold 4/15 | Test subject S5 ===
  -> Acc=0.9178 F1=0.8950 AUROC=0.9663 | fold_time=170.8s

=== CNN-LSTM Fold 5/15 | Test subject S6 ===
  -> Acc=0.8897 F1=0.8711 AUROC=0.9615 | fold_time=167.1s

=== CNN-LSTM Fold 6/15 | Test subject S7 ===
  -> Acc=0.9931 F1=0.9916 AUROC=0.9998 | fold_time=132.0s

=== CNN-LSTM Fold 7/15 | Test subject S8 ===
  -> Acc=0.9932 F1=0.9919 AUROC=1.0000 | fold_time=202.9s

=== CNN-LSTM Fold 8/15 | Test subject S9 ===
  -> Acc=0.6621 F1=0.6311 AUROC=0.7513 | fold_time=868.8s

=== CNN-LSTM Fold 9/15 | Test subject S10 ===
  -> Acc=0.8667 F1=0.8347 AUROC=0.9033 | fold_time=194.3s

=== CNN-LSTM Fold 10/15 | Test subject S11 ===
  -> Acc=1.0000

In [10]:
total_time = time.time() - start_total

print(
    f"\nTOTAL TIME: "
    f"{total_time/60:.1f} minutes"
)


TOTAL TIME: 90.2 minutes


In [11]:
results_lstm_df = pd.DataFrame(
    all_results_lstm
)

results_lstm_df.to_csv(
    "../results/cnn_lstm_full_modality_results.csv",
    index=False
)

with open(
    "../results/cnn_lstm_predictions_full.pkl",
    "wb"
) as f:
    pickle.dump(
        all_predictions_lstm,
        f
    )

print(
    "Results saved successfully."
)

Results saved successfully.


In [12]:
print("\n--- CNN-LSTM Full-Modality Summary ---")

print(
    f"Mean Accuracy: "
    f"{results_lstm_df['accuracy'].mean():.4f} "
    f"± {results_lstm_df['accuracy'].std():.4f}"
)

print(
    f"Mean Macro-F1: "
    f"{results_lstm_df['f1_macro'].mean():.4f} "
    f"± {results_lstm_df['f1_macro'].std():.4f}"
)

print(
    f"Mean AUROC: "
    f"{results_lstm_df['auroc'].mean():.4f} "
    f"± {results_lstm_df['auroc'].std():.4f}"
)

print(
    "Per-fold checkpoints saved "
    "in ../results/models/"
)


--- CNN-LSTM Full-Modality Summary ---
Mean Accuracy: 0.8970 ± 0.0941
Mean Macro-F1: 0.8786 ± 0.1052
Mean AUROC: 0.9481 ± 0.0759
Per-fold checkpoints saved in ../results/models/


In [13]:
handcrafted_results = pd.read_csv("../results/handcrafted_loso_full.csv")
cnn_results = pd.read_csv("../results/cnn_full_modality_results.csv")
cnn_lstm_results = pd.read_csv("../results/cnn_lstm_full_modality_results.csv")

# Keep only SVM rows
svm_results = handcrafted_results[
    handcrafted_results["classifier"] == "SVM"
].copy()

comparison = pd.DataFrame({
    "Model": [
        "Handcrafted + SVM",
        "CNN",
        "CNN-LSTM"
    ],

    "Mean_F1": [
        svm_results["macro_f1"].mean(),
        cnn_results["f1_macro"].mean(),
        cnn_lstm_results["f1_macro"].mean()
    ],

    "Std_F1": [
        svm_results["macro_f1"].std(),
        cnn_results["f1_macro"].std(),
        cnn_lstm_results["f1_macro"].std()
    ],

    "Mean_AUROC": [
        svm_results["auroc"].mean(),
        cnn_results["auroc"].mean(),
        cnn_lstm_results["auroc"].mean()
    ],

    "Mean_Accuracy": [
        svm_results["accuracy"].mean(),
        cnn_results["accuracy"].mean(),
        cnn_lstm_results["accuracy"].mean()
    ],

    "Mean_TrainTime_sec": [
        np.nan,
        cnn_results["train_time_sec"].mean(),
        cnn_lstm_results["train_time_sec"].mean()
    ]
})

print(comparison)

comparison.to_csv(
    "../results/three_way_comparison.csv",
    index=False
)

               Model   Mean_F1    Std_F1  Mean_AUROC  Mean_Accuracy  \
0  Handcrafted + SVM  0.874648  0.077102    0.972048       0.895231   
1                CNN  0.849155  0.148726    0.970529       0.893087   
2           CNN-LSTM  0.878606  0.105155    0.948111       0.896992   

   Mean_TrainTime_sec  
0                 NaN  
1          499.290056  
2          293.263992  
